Metabolomics data analysis in B. pseudocatenulatum-treated mice

Prior to statistical analysis, raw peak areas of annotated known metabolites were normalized via median centering to reduce technical variability. Briefly, the peak intensity of each metabolite was divided by the median intensity of all metabolites within the same sample for centering correction, which minimized variations in instrumental response and injection volume across samples. The final relative abundance of each metabolite was defined as the normalized peak intensity ratio (unitless), which reflects the relative level of the metabolite in each sample.

In [ ]:
rm(list = ls())
library(ppcor)
library(psych)
library(reshape2)
library(igraph)
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(Maaslin2)
library(car)
library(tidyverse)
library(Seurat)
library(ggpubr)
library(pheatmap)
library(tidyfst)
library(scales)
library(RColorBrewer)
library(fields)

data<-read.csv("MetaboliteRawdata.csv",row.names = 1,check.names = F)
df_metadata<-read.csv("Metadata.csv"
)
Metabolite_metadata<-read.csv("MetaboliteInfo.csv")%>%
  filter(Name!="")
data<-data[Metabolite_metadata$UniqueID,]
norm_mat <- apply(data, 2, function(x) (x / median(x, na.rm=T)))

dim(norm_mat)

write.csv(norm_mat,"data_normalized_matrix.csv")

PCA

In [ ]:
rm(list = ls())
library(psych)
library(reshape2)
library(igraph)
library(ggplot2)
library(Hmisc)
library(patchwork)
library(limma)
library(car)
library(Seurat)
library(ggpubr)
library(pheatmap)
library(tidyfst)
library(scales)
library(RColorBrewer)
library(fields)
library(ggsci)
library(dplyr)
library(microbiome)
library(vegan)
library(ecodist)
library(tidyverse)
library(ppcor)
library(compositions)
library(mice)
library(mgcv)

data<-read.csv("data_normalized_matrix.csv",row.names = 1,check.names = F)
meta<-read.csv("Metadata.csv"
)
Metabolite_metadata<-read.csv("MetaboliteInfo.csv")%>%
  filter(Name!="")
data<-data.frame(log2(data+1),check.names = F)
  pca_res <- prcomp(t(data))
  pca_data <- data.frame(pca_res$x,check.names = F)
  pca_data[1:5,1:5]
  
  pca_data <- pca_data[,1:10]
  pca_data$Sample <- rownames(pca_data)
  head(pca_data,5)
  head(meta,5)
  pca_data <- merge(pca_data, meta, by.x = 'Sample', by.y="SAMPLEID")
  
  write.csv(pca_data, 'PCs_info.csv')

  loadings <- pca_res$rotation %>% 
    as.data.frame() %>% 
    dplyr::select(PC1, PC2)

p5 <- ggplot(pca_data, aes(PC1, PC2)) +
  geom_point(size=2, aes(color = Group)) +stat_ellipse(aes(colour =Group))+
  xlab(paste('PC1: ', round(summary(pca_res)$importance[2,1]*100, 1), '% variance')) +
  ylab(paste('PC2: ', round(summary(pca_res)$importance[2,2]*100, 1), '% variance')) +
  theme_bw() +
  theme(axis.text = element_text(color = 'black'), axis.ticks = element_line(color = 'black'))
p5

The Wilcoxon rank-sum test was employed to calculate the P value, while fold changes were determined by computing the ratio of the mean normalized peak intensity in the B. pseudocatenulatum-treated group to that in the vehicle control group.

In [ ]:
rm(list=ls())
library(ggsci)
library(dplyr)
library(patchwork)
library(microbiome)
library(vegan)
library(ecodist)
library(tidyverse)
library(ggplot2)
library(ppcor)
library(compositions)
library(mice)
library(mgcv)

data<-read.csv("data_normalized_matrix.csv",row.names = 1,check.names = F)
df_metadata<-read.csv("Metadata.csv"
)
Metabolite_metadata<-read.csv("MetaboliteInfo.csv")%>%
  filter(Name!="")
data<-data[Metabolite_metadata$UniqueID,]
data<-data.frame(t(data),check.names = F)

data$SAMPLEID <- rownames(data)

data_4_core<-merge(df_metadata,data,by="SAMPLEID")

rownames(data_4_core)<-data_4_core$SAMPLEID

df_metadata_young<-df_metadata%>%
  filter(Group =="Veh")
df_metadata_old<-df_metadata%>%
  filter(Group =="BP")

meta<-df_metadata[c(rownames(df_metadata_young),rownames(df_metadata_old)),]

meta_data<-data_4_core
result_list<-list()
n=1
for (i in seq(3,dim(data_4_core)[2]-1,1)){
  print(names(data_4_core)[i])
  meta_data[[i]]
  tmp <- meta_data
  names(tmp)[i] <- "tmp_feature"
  
  Young <- tmp %>%dplyr::filter(SAMPLEID  %in% df_metadata_young$SAMPLEID)%>%
    filter(!(is.na(tmp_feature)))
  Old <- tmp %>%dplyr::filter(SAMPLEID  %in% df_metadata_old$SAMPLEID)%>%
    filter(!(is.na(tmp_feature)))
  if(min(dim(Young)[1],dim(Old)[1])>0){
    result=data.frame(p.value = wilcox.test(Young[[i]],Old[[i]])$p.value,
                      FoldChange = mean(Old[[i]])/mean(Young[[i]]),
                      Value_Y=mean(Young[[i]]),
                      Value_O=mean(Old[[i]]),
                      N_Y=dim(Young)[1],
                      N_O=dim(Old)[1])}else{
                        result=data.frame(p.value = NA,
                                          FoldChange =NA,
                                          Value_Y=mean(Young[[i]]),
                                          Value_O=mean(Old[[i]]),
                                          N_Y=dim(Young)[1],
                                          N_O=dim(Old)[1])}  
  if (dim(result)[1]>0){
    result_list[[n]]=result
    result_list[[n]]$measID=colnames(meta_data)[i]
    n=n+1
  }
}

print(n)
result_clinical_data<-result_list[[1]]

for (i in seq(2,length(result_list),1)){
  result_clinical_data<-rbind(result_clinical_data,result_list[[i]])
}

result_clinical_data
padj=data.frame(padj=p.adjust(result_clinical_data$p.value,method = "BH"))
final<-cbind(result_clinical_data,padj)%>%arrange(FoldChange)
dim(final)

df<-merge(final,Metabolite_metadata,by.x="measID",by.y="UniqueID")
df